In [ ]:
import json
import os
import sys

import polars as pl

sys.path.append("/workspace")

from drsd.reader import MINDsmallReader

pl.Config(tbl_rows=5)
os.makedirs("/workspace/processed", exist_ok=True)

In [ ]:
reader = MINDsmallReader()

### Target News

In [ ]:
news_df = (
    pl.concat([reader.get_news_df("train"), reader.get_news_df("dev")])
    .select("news_id", "title_entities", "abstract_entities")
    .unique()
    .sort("news_id")
)
assert news_df.get_column("news_id").is_unique().all()
news_df

### Extract News Entities

In [ ]:
output_df = news_df.select(
    "news_id",
    pl.concat_list(
        pl.col("title_entities", "abstract_entities").map_elements(
            lambda x: [f"{d['Label']} ({d['Type']})" for d in json.loads(x)] if x != "" else [],
            return_dtype=pl.List(pl.String),
        )
    )
    .list.unique()
    .list.sort()
    .alias("entities"),
)
output_df

### Output DataFrame

In [ ]:
output_df.write_parquet("/workspace/processed/news_entity.parquet")